# Лекција 09 - Образац метакогниције


## Постављање

Овај нотебоок демонстрира дизајн шаблона Метакогниције користећи Microsoft Agent Framework.

**Предуслови:**
- Azure OpenAI распоређивање конфигурисано преко променљивих окружења
- Azure CLI аутентификован (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Шта је метакогница?

Метакогница је **размишљање о размишљању**. У контексту AI агената, то значи креирање агената који могу:

- **Саморефлексија** о својим излазима и процесу резоновања
- **Препознавање грешака** и опоравак на елегантан начин уместо тихог пада
- **Процена** да ли су њихови одговори потпуни и корисни
- **Прилагођавање** своје стратегије када почетни приступ не функционише (нпр. прелазак на резервни систем)

Метакогнитивни агент не само да одговара на питања — он прати своје перформансе и подешава се у ходу.


## Примарни и резервни алати

Уобичајени мета-когнитивни образац је **стратегија резерве**. Агент прво покушава са примарним алатом; ако не успе (нпр. грешка 404), агент препознаје неуспех и транспарентно прелази на резервни алат.

Ово одражава системе из стварног света где примарне услуге могу бити недоступне, а агенти морају сами дијагностиковати проблем пре него што изаберу алтернативни пут.

Испод дефинишемо два алата за претрагу летова:
- **Примарни** — покрива Париз, Токио и Барселону
- **Резервни** — покрива Берлин, Сиднеј и Њујорк


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Саморефлектујући агент са опоравком од грешака

Испод наведени агент добија инструкције да прво покуша са примарним системом лета, препозна грешке и транспарентно пређе на резервни систем. Након сваког одговора кратко самоевалуира да ли је у потпуности одговорио на корисничко питање.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Образац за самоевалуацију

Још један аспект метакогниције је **самоевалуација**: одвојени агент (или исти агент у другом пролазу) прегледа одговор ради потпуности, тачности и корисности.

Испод правимо агента `ResponseEvaluator` који оцењује одговоре туристичког агента на три димензије.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Резиме

У овој лекцији сте научили како да правите **метакогнитивне агенте** користећи Microsoft Agent Framework:

- **Саморефлексија**: Агенти који прате свој сопствени процес резоновања и транспарентно комуницирају шта се десило.
- **Опоравак од грешака уз резервне алтернативе**: Патерн главни + резервни инструмент где агент препознаје грешке (нпр. 404 грешке) и аутоматски покушава алтернативни извор.
- **Самоевалуација**: Посебан агент за оцену који оцењује одговоре по потпуности, тачности и корисности.

Ови патерни чине агенте робуснијим, транспарентнијим и поузданијим — критичним особинама за продукцијске примене.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
